# Problem 6: Array Jobs

---

> Place all files for this problem in a directory called `problem_6`.

## Overview

In Module 3, you implemented the Bible Code ELS search sequentially (Problem 4) and with Python `multiprocessing` (Problem 5). Both approaches ran on a single machine. In this problem, you will use **SLURM array jobs** to distribute the same workload across the cluster.

An array job lets you submit a single `.sbatch` script that SLURM expands into many independent tasks. Each task receives a unique integer through the environment variable `$SLURM_ARRAY_TASK_ID`. The key insight is that these IDs are **sequential integers** — and so are ELS strides. That means we can use the task ID directly as the stride value, with no extra mapping or arithmetic required.

```
--array=2-100  →  SLURM creates 99 tasks with IDs 2, 3, 4, ..., 100
                  Each task ID is already the stride we want to search.
```

This is a common pattern in HPC: whenever your problem is parameterized by an integer (a stride, a chromosome number, a simulation seed, a frame index), array jobs let you run all values simultaneously with almost no code changes.


## Part A: Adapt Your Script

Refactor your ELS search from Module 3 Problem 4 so it can be run from the command line and accepts a **single stride** and a **text identifier** as arguments:

```bash
python3 els_search.py --text bible --stride 47
```

This should run the ELS extraction for only stride 47 against the Bible, and write any matches to an output file that includes the stride in the filename (e.g., `results/bible_stride_047.txt`).

A few tips for converting your notebook code to a command-line script:

- Use `argparse` or `sys.argv` to handle command-line arguments
- Make sure file paths work relative to where the job runs on the cluster, not your local machine
- Test it interactively first with `sinteractive` before submitting as a batch job


## Part B: Submit an Array Job

Write a batch script called `els_array.sbatch` that uses `$SLURM_ARRAY_TASK_ID` as the stride:

```bash
#!/bin/bash
#SBATCH --job-name=els_array
#SBATCH --account=mpcs56430
#SBATCH --partition=caslake
#SBATCH --array=2-100
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --mem=2G
#SBATCH --time=00:10:00
#SBATCH --output=logs/els_%A_%a.out
#SBATCH --error=logs/els_%A_%a.err

python3 els_search.py --text bible --stride $SLURM_ARRAY_TASK_ID
```

Notice that `$SLURM_ARRAY_TASK_ID` takes values 2 through 100 — the first 99 strides from the original problem. Each stride runs as its own independent job on the cluster. There is no coordination between tasks because no stride depends on any other stride.

Before submitting:

1. Create the `logs/` directory so SLURM has somewhere to write output files.
2. Make sure your data files (texts, word lists) are accessible from the cluster.

Submit the job and observe:

```bash
sbatch els_array.sbatch
squeue --user=<your_cnetid>
```

In `answers.md`, record:

   - How do array task job IDs appear in `squeue`?
   - How many tasks ran simultaneously versus waiting in the queue?
   - Pick a few `.out` files from the `logs/` directory and verify the correct stride was used.


## Part C: Controlling Array Parallelism

Submitting many tasks at once may flood the scheduler or consume your allocation quickly. SLURM lets you **throttle** how many array tasks run at the same time using the `%` operator:

```bash
#SBATCH --array=2-100%20
```

This still creates 99 tasks, but only 20 run concurrently — the rest wait in the queue until a running task finishes.

1. Resubmit your array job with a concurrency limit (e.g., `%10` or `%20`).
2. Watch the queue with `squeue` and observe how tasks cycle through.
3. In `answers.md`, explain why throttling can be important on a shared cluster. What would happen if every student in the class submitted 99 unrestricted tasks at the same time?


## Part D: Combining Results

After the array job completes, your results are spread across up to 99 separate files. Write a short script called `combine_results.py` (or a shell script) that merges all per-stride output files into a single results file matching the format from your original Module 3 solution.

Verify that the combined output contains the same matches as your original sequential run.


## Part E: Speedup Comparison

Using the **Bible** text as your benchmark, collect wall times for each approach and calculate the speedup. Use your timing results from Module 3 Problem 5 for the serial and multiprocessing rows. For the array job, measure from when the first task started to when the last task completed (use `sacct`).

| Approach                         | Wall Time | Speedup ($S = T_{serial} / T_{approach}$) |
| -------------------------------- | --------- | ------------------------------------------ |
| Serial (Module 3 Ex. 4)          |           |                                            |
| Multiprocessing (Module 3 Ex. 5) |           |                                            |
| Array Job (Part B)               |           |                                            |

In `answers.md`, reflect on the results:

   - Which approach achieved the best speedup?
   - What overhead exists in the array job approach that doesn't exist in multiprocessing? (Think about job scheduling, file I/O, result merging.)
   - How does the array job approach differ fundamentally from Python `multiprocessing`? Consider where the parallelism happens — within a single node's cores versus across many independent nodes — and what that means for scaling to even larger problems.
   - Could you combine both approaches — multiprocessing *within* each array task — for even better performance? When might that be worthwhile?


## Deliverables

Your `problem_6` directory should contain:

```
problem_6/
├── answers.md
├── els_search.py
├── els_array.sbatch
├── combine_results.py
└── logs/           (include a few sample .out files)
```

**Submitting.** Put everything for this problem in `problem_6/` in your repository, with your results easy to find: your scripts, the output files your answers refer to, and `answers.md`. You can work in your clone on Midway3 and commit and push from there, or copy your work back to your laptop (for example with `scp`) and push from there. Either is fine.
